# Laboratorio 7 - Regresión Logística

In [4]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import random
import statsmodels.api as sm
import statsmodels.stats.diagnostic as smd
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split
import seaborn as sns
import statsmodels.api as sm
import scipy.stats as stats
import statsmodels.stats.diagnostic as diag
from sklearn.linear_model import LogisticRegression
import pyreadr
import random

#Metrics
from sklearn.metrics import make_scorer, accuracy_score,precision_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score ,precision_score,recall_score,f1_score
from sklearn.impute import SimpleImputer

In [5]:
random.seed(42)
np.random.seed(42)

result = pyreadr.read_r('listings.RData')
df = result[list(result.keys())[0]]

### Carga de datos

In [6]:
result = pyreadr.read_r('listings.RData')
df = result['listings']

# Limpiar price
df['price'] = df['price'].str.replace('$', '', regex=False).str.replace(',', '', regex=False)
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df = df.dropna(subset=['price'])

df.shape

(76246, 80)

In [7]:
cols_to_keep = [
    'price', 'accommodates', 'bathrooms', 'bedrooms', 'beds',
    'room_type', 'minimum_nights', 'number_of_reviews',
    'review_scores_rating', 'reviews_per_month'
]

df_model = df[cols_to_keep].copy().dropna()

# Encoding de room_type
df_model = pd.get_dummies(df_model, columns=['room_type'], drop_first=True)

In [8]:
# Mismos bins que labs anteriores
bins   = [0, 100, 250, float('inf')]
labels = ['economica', 'media', 'cara']

df_model['precio_cat'] = pd.cut(df_model['price'], bins=bins, labels=labels)

# Variable dicotómica para cada categoría
df_model['es_cara']      = (df_model['precio_cat'] == 'cara').astype(int)
df_model['es_media']     = (df_model['precio_cat'] == 'media').astype(int)
df_model['es_economica'] = (df_model['precio_cat'] == 'economica').astype(int)

print("Distribución de categorías:")
print(df_model['precio_cat'].value_counts())
print("\nVerificación de variables dicotómicas:")
print(df_model[['es_cara','es_media','es_economica']].head(10))

Distribución de categorías:
precio_cat
media        30411
cara         20841
economica    11470
Name: count, dtype: int64

Verificación de variables dicotómicas:
   es_cara  es_media  es_economica
0        0         0             1
1        0         1             0
2        0         0             1
3        0         1             0
4        0         0             1
5        0         0             1
6        1         0             0
7        0         1             0
8        0         1             0
9        0         1             0


In [9]:
X = df_model.drop(columns=['price', 'precio_cat', 'es_cara', 'es_media', 'es_economica'])
y_cara      = df_model['es_cara']
y_media     = df_model['es_media']
y_economica = df_model['es_economica']

# Split 70/30 con misma semilla
X_train, X_test, y_train_cara, y_test_cara = train_test_split(
    X, y_cara, test_size=0.3, random_state=42, stratify=y_cara
)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("\nBalance en entrenamiento:")
print(y_train_cara.value_counts(normalize=True).round(3))

Train: (43905, 11) | Test: (18817, 11)

Balance en entrenamiento:
es_cara
0    0.668
1    0.332
Name: proportion, dtype: float64
